In [2]:
import boto3, botocore
from botocore.exceptions import ClientError
from dotenv import load_dotenv
import os, time, json, shutil, subprocess, zipfile
from datetime import date
from pathlib import Path

from misc import load_from_yaml, save_to_yaml
import s3, iam, lftn, glue, lambdafn, rds, dynamodb as ddb, eventbridge as event

load_dotenv(".env")

from mylogger import CustomLogger
logger = CustomLogger()

In [3]:
ACCOUNT_ID        = os.environ['AWS_ACCOUNT_ID_ROOT']
REGION            = os.environ['AWS_DEFAULT_REGION']
VPC_ID            = os.environ['AWS_DEFAULT_VPC']
SECURITY_GROUP_ID = os.environ['AWS_DEFAULT_SG_ID']
SUBNET_IDS        = SUBNET_IDS = os.environ["AWS_DEFAULT_SUBNET_IDS"].split(":")
SUBNET_ID         = SUBNET_IDS[0]
logger.info(SUBNET_ID)

INFO: 2025-09-24 19:25:28 [712755687.py:7] ("subnet-0085102d26a994304" "subnet-0de9c2ad843d1f61f" "subnet-0af3f60a4199e12cb" "subnet-0aef213c44bc3a23f" "subnet-0a261cdcd7d81e208" "subnet-0ce632f3acb0d440d")


In [4]:
sts_client           = boto3.client('sts')
rds_client           = boto3.client('rds')
iam_client           = boto3.client('iam')
s3_client            = boto3.client('s3')
lakeformation_client = boto3.client('lakeformation')
ec2_client           = boto3.client('ec2', region_name=REGION)
ec2_resource         = boto3.resource('ec2', region_name=REGION)
dynamodb_client      = boto3.client('dynamodb')
events_client        = boto3.client('events')
lambda_client        = boto3.client('lambda')
glue_client          = boto3.client('glue')
databrew_client      = boto3.client('databrew')

#### [S3 Boto3 API](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/s3.html)

1. s3_client`.close( ... )`
1. s3_client`.copy( ... )`
1. s3_client`.copy_object( ... )`
1. s3_client`.create_bucket( ... )`
1. s3_client`.create_session( ... )`
1. s3_client`.delete_bucket( ... )`
1. s3_client`.delete_bucket_cors( ... )`
1. s3_client`.delete_bucket_encryption( ... )`
1. s3_client`.delete_bucket_metrics_configuration( ... )`
1. s3_client`.delete_bucket_policy( ... )`
1. s3_client`.delete_object( ... )`
1. s3_client`.delete_objects( ... )`
1. s3_client`.download_file( ... )`
1. s3_client`.download_fileobj( ... )`
1. s3_client`.get_bucket_acl( ... )`
1. s3_client`.get_bucket_cors( ... )`
1. s3_client`.get_bucket_encryption( ... )`
1. s3_client`.get_bucket_location( ... )`
1. s3_client`.get_bucket_logging( ... )`
1. s3_client`.get_bucket_metrics_configuration( ... )`
1. s3_client`.get_bucket_notification( ... )`
1. s3_client`.get_bucket_notification_configuration( ... )`
1. s3_client`.get_bucket_policy( ... )`
1. s3_client`.get_bucket_policy_status( ... )`
1. s3_client`.get_bucket_request_payment( ... )`
1. s3_client`.get_bucket_versioning( ... )`
1. s3_client`.get_object( ... )`
1. s3_client`.get_object_acl( ... )`
1. s3_client`.get_object_attributes( ... )`
1. s3_client`.get_public_access_block( ... )`
1. s3_client`.head_bucket( ... )`
1. s3_client`.head_object( ... )`
1. s3_client`.list_bucket_metrics_configurations( ... )`
1. s3_client`.list_buckets( ... )`
1. s3_client`.list_directory_buckets( ... )`
1. s3_client`.list_object_versions( ... )`
1. s3_client`.list_objects( ... )`
1. s3_client`.list_objects_v2( ... )`
1. s3_client`.put_bucket_acl( ... )`
1. s3_client`.put_bucket_cors( ... )`
1. s3_client`.put_bucket_encryption( ... )`
1. s3_client`.put_bucket_logging( ... )`
1. s3_client`.put_bucket_metrics_configuration( ... )`
1. s3_client`.put_bucket_notification( ... )`
1. s3_client`.put_bucket_notification_configuration( ... )`
1. s3_client`.put_bucket_policy( ... )`
1. s3_client`.put_bucket_request_payment( ... )`
1. s3_client`.put_bucket_tagging( ... )`
1. s3_client`.put_bucket_versioning( ... )`
1. s3_client`.put_object( ... )`
1. s3_client`.put_object_acl( ... )`
1. s3_client`.put_object_retention( ... )`
1. s3_client`.restore_object( ... )`
1. s3_client`.select_object_content( ... )`
1. s3_client`.upload_file( ... )`
1. s3_client`.upload_fileobj( ... )`

# [S3](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/s3.html)

In [ ]:
# [method for method in dir(s3_client) if not method.startswith("_")]

##### Create a Bucket

In [7]:
S3_BUCKET_DATALAKE = "htech-datalake-bkt"
S3_BUCKET_GLUE_ASSETS = "htech-glue-assets-bkt"

In [ ]:
s3 = boto3.resource('s3')
bucket1 = s3.Bucket(S3_BUCKET_DATALAKE)
bucket2 = s3.Bucket(S3_BUCKET_GLUE_ASSETS)

# Delete all objects in the bucket
bucket1.objects.all().delete()
bucket2.objects.all().delete()

# Delete all object versions (if versioning is enabled)
# bucket1.object_versions.all().delete()
# bucket2.object_versions.all().delete()

# Finally, delete the bucket
bucket1.delete()
bucket2.delete()

In [ ]:
acl = 'public-read'                         # Set the ACL (e.g., 'private', 'public-read')
enable_versioning = False                   # Enable versioning
enable_encryption = False                   # Enable server-side encryption

folders1 = ['raw/customers/', 'cleansed/customers/']
folders2 = ['temporary/', 'sparkHistoryLogs/']

In [ ]:
s3_client.create_bucket(Bucket=S3_BUCKET_DATALAKE)
s3_client.create_bucket(Bucket=S3_BUCKET_GLUE_ASSETS)

##### Put Object (Upload)

In [ ]:
# s3_client.put_object(Bucket='my-bucket', Key='new_file.txt', Body=b'Hello, World!')

In [ ]:
s3_client.put_object(Bucket=S3_BUCKET_DATALAKE, Key="processed/sales/")

In [ ]:
[s3_client.put_object(Bucket=S3_BUCKET_DATALAKE, Key=folder) for folder in folders1]
[s3_client.put_object(Bucket=S3_BUCKET_GLUE_ASSETS, Key=folder) for folder in folders2]

##### List All Buckets: Lists all the buckets in your S3 account.

In [ ]:
response = s3_client.list_buckets()
# print(json.dumps(response, indent=4, default=str))
for bucket in response['Buckets']:
    print(f'Bucket: {bucket["Name"]}')

##### Upload a File to S3: Uploads a file to a specified S3 bucket.

In [17]:
# s3_client.upload_file('./data/customers.csv', S3_BUCKET_DATALAKE, 'raw/customers/customers.csv')
s3_client.upload_file(
    "../aws_glue/data/sales.csv", S3_BUCKET_DATALAKE, "MISC/sales/sales.csv"
)

#####  Download a File from S3: Downloads a file from an S3 bucket to a local directory.

In [ ]:
s3_client.download_file(S3_BUCKET_DATALAKE, 'raw/customers/customers.csv', './data/downloaded_file.csv')

##### List Objects in a Bucket: Lists all objects within a bucket.

In [5]:
response = s3_client.list_objects_v2(Bucket="datalake-bkt-08242025", Prefix="processed/customers")
logger.info(response)
for obj in response.get('Contents', []):
    logger.info(f"Object: {obj['Key']}")

INFO: 2025-09-05 14:08:52 [3036384922.py:2] {
    "ResponseMetadata": {
        "RequestId": "CYSCE8S9MJSQHM22",
        "HostId": "qaTacOLLZ7TnvIrOmvDoUZvJwfDNwClTG4MoNNVwzIVEXGqUpXWV2topyFiBwMdb/8h4D08MlnH3uRlTbSPFxA==",
        "HTTPStatusCode": 200,
        "HTTPHeaders": {
            "x-amz-id-2": "qaTacOLLZ7TnvIrOmvDoUZvJwfDNwClTG4MoNNVwzIVEXGqUpXWV2topyFiBwMdb/8h4D08MlnH3uRlTbSPFxA==",
            "x-amz-request-id": "CYSCE8S9MJSQHM22",
            "date": "Fri, 05 Sep 2025 19:08:53 GMT",
            "x-amz-bucket-region": "us-east-1",
            "content-type": "application/xml",
            "transfer-encoding": "chunked",
            "server": "AmazonS3"
        },
        "RetryAttempts": 0
    },
    "IsTruncated": false,
    "Contents": [
        {
            "Key": "processed/customers/part-00000-33338bd1-da4b-4d4d-8857-81083bdda421-c000.snappy.parquet",
            "LastModified": "2025-09-05 18:55:21+00:00",
            "ETag": "\"dea376d4cde2da58ce00e05e9e2fa29e-1\""

In [ ]:
# Check if the response contains any contents
if 'Contents' in response:
    for obj in response['Contents']:
        print(obj['Key'])
else:
    print(f"No objects found with prefix 'r' in bucket '{S3_BUCKET_DATALAKE}'.")


##### Get Object (Download): Retrieves an object directly from S3.

In [ ]:
response = s3_client.get_object(Bucket=S3_BUCKET_DATALAKE, Key='raw/customers/customers.csv')
print(response)
content = response['Body'].read()
print(content.decode('utf-8'))

##### Copy an Object to Another Bucket: Copies an object from one bucket to another.

In [ ]:
copy_source = {'Bucket': S3_BUCKET_DATALAKE, 'Key': 'raw/customers/customers.csv'}
s3_client.copy_object(CopySource=copy_source, Bucket=S3_BUCKET_GLUE_ASSETS, Key='copied-file.txt')

##### Delete an Object: Deletes an object from a bucket.

In [ ]:
s3_client.delete_object(Bucket=S3_BUCKET_GLUE_ASSETS, Key='copied-file.txt')

##### Put Configuration

In [ ]:
# The configuration 'EventBridgeConfiguration': {} to route all S3 events (like s3:ObjectCreated:* events) to EventBridge.

# Define the bucket notification configuration for EventBridge
notification_configuration = {
    'EventBridgeConfiguration': {}
}

# Put bucket notification configuration
response = s3_client.put_bucket_notification_configuration(
    Bucket=S3_BUCKET_DATALAKE,
    NotificationConfiguration=notification_configuration
)

print("Bucket notification configuration response:")
print(response)

In [ ]:
# Add S3 trigger to the Lambda function
response = s3_client.put_bucket_notification_configuration(
    Bucket=S3_BUCKET_DATALAKE,
    NotificationConfiguration={
        'LambdaFunctionConfigurations': [
            {
                'LambdaFunctionArn': LFN_CRAWLER_ARN,
                'Events': [
                    's3:ObjectCreated:*'  # Trigger Lambda on object creation
                ],
                'Filter': {
                    'Key': {
                        'FilterRules': [
                            {
                                'Name': 'prefix',
                                'Value': 'raw/customers/'  # Trigger only on this prefix
                            },
                        ]
                    }
                }
            }
        ]
    }
)

##### Generate a Pre-signed URL: Generates a pre-signed URL for secure access to a file.

In [ ]:
url = s3_client.generate_presigned_url('get_object',
                                Params={'Bucket': 'my-bucket', 'Key': 'uploaded_file.txt'},
                                ExpiresIn=3600)
print(f"Presigned URL: {url}")

##### Check if an Object Exists: Retrieves metadata from an object without returning the object itself, useful for checking if a file exists.

In [ ]:
try:
    s3_client.head_object(Bucket='my-bucket', Key='file.txt')
    print("Object exists")

except ClientError:
    print("Object does not exist")


##### Delete a Bucket: Deletes an empty bucket.

In [ ]:
s3 = boto3.resource("s3")
bucket1 = s3.Bucket("glue-assets-bkt")
bucket2 = s3.Bucket("datalake-bkt")
bucket3 = s3.Bucket("glue-temp-bkt")

# Delete all objects in the bucket
bucket1.objects.all().delete()
bucket2.objects.all().delete()
bucket3.objects.all().delete()

# Delete all object versions (if versioning is enabled)
# bucket1.object_versions.all().delete()
# bucket2.object_versions.all().delete()

# Finally, delete the bucket
bucket1.delete()
bucket2.delete()
bucket3.delete()


{'ResponseMetadata': {'RequestId': '74XSDJGFR385R07P',
  'HostId': '0YFL2/GrjHb710y+CjyoU5C+7CXefI5Yy5dQEIDcqPDKlde5JDUy432NV+YbeFbR68HAEaRX3dNi8k3jQHqFMx7Urhvbl6YE',
  'HTTPStatusCode': 204,
  'HTTPHeaders': {'x-amz-id-2': '0YFL2/GrjHb710y+CjyoU5C+7CXefI5Yy5dQEIDcqPDKlde5JDUy432NV+YbeFbR68HAEaRX3dNi8k3jQHqFMx7Urhvbl6YE',
   'x-amz-request-id': '74XSDJGFR385R07P',
   'date': 'Thu, 25 Sep 2025 00:26:49 GMT',
   'server': 'AmazonS3'},
  'RetryAttempts': 0}}

In [ ]:
s3_client.delete_bucket(Bucket='my-bucket')

##### Creating a Public S3 Bucket

In [ ]:
# Initialize boto3 S3 client
s3 = boto3.client('s3')

# Replace with your desired bucket name and region
bucket_name = "your-public-bucket-name"
region = "us-east-1"  # Specify your AWS region

# Step 1: Create the S3 bucket
if region == "us-east-1":
    response = s3.create_bucket(Bucket=bucket_name)
else:
    response = s3.create_bucket(
        Bucket=bucket_name,
        CreateBucketConfiguration={"LocationConstraint": region},
    )
print(f"Bucket '{bucket_name}' created successfully.")

In [ ]:

# Step 2: Set the bucket to be publicly accessible using a bucket policy
bucket_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "PublicReadGetObject",
            "Effect": "Allow",
            "Principal": "*",
            "Action": "s3:GetObject",
            "Resource": f"arn:aws:s3:::{S3_BUCKET_DATALAKE}/*",
        }
    ],
}

# Convert the policy to a JSON string
bucket_policy_json = json.dumps(bucket_policy)

# Apply the bucket policy
s3_client.put_bucket_policy(Bucket=S3_BUCKET_DATALAKE, Policy=bucket_policy_json)
print(f"Public read access policy applied to bucket '{S3_BUCKET_DATALAKE}'.")

In [ ]:
# Step 3: Enable bucket ACL for public read access (optional, not needed if bucket policy is applied)
s3_client.put_bucket_acl(Bucket=S3_BUCKET_GLUE_ASSETS, ACL="public-read")
print(f"Bucket ACL set to 'public-read' for '{S3_BUCKET_GLUE_ASSETS}'.")

print("Public S3 bucket setup completed successfully!")

In [ ]:
import boto3
from botocore.config import Config
from botocore.exceptions import NoCredentialsError, PartialCredentialsError

# Configure S3 client for anonymous access
try:
    public_s3_client = boto3.client(
        's3',
        config=Config(signature_version='s3v4'),
        aws_access_key_id=None,
        aws_secret_access_key=None,
    )

    # Step 1: List objects in the public bucket
    print(f"Listing objects in the bucket '{S3_BUCKET_GLUE_ASSETS}':")
    response = public_s3_client.list_objects_v2(Bucket=S3_BUCKET_GLUE_ASSETS)
    for obj in response.get('Contents', []):
        print(f"- {obj['Key']}")
    
    response = s3_client.get_object(Bucket=S3_BUCKET_GLUE_ASSETS, Key='copied-file.txt')
    print(response)
    content = response['Body'].read()
    print(content.decode('utf-8'))


except NoCredentialsError:
    print("Error: No credentials provided. Anonymous access failed.")
except PartialCredentialsError:
    print("Error: Partial credentials provided. Anonymous access failed.")
except Exception as e:
    print(f"An error occurred: {e}")


##### downloads a file from a public S3 bucket using Boto3 and/or standard HTTP request methods

In [ ]:
import boto3

# Create anonymous session
session = boto3.session.Session()
s3 = session.client("s3", config=boto3.session.Config(signature_version="unsigned"))

bucket_name = "commoncrawl"
object_key = "crawl-data/CC-MAIN-2024-10/warc.paths.gz"
output_file = "warc.paths.gz"

try:
    s3.download_file(bucket_name, object_key, output_file)
    print(f"✅ Downloaded {object_key} from {bucket_name}")
except Exception as e:
    print(f"❌ Failed to download: {e}")


In [ ]:
import urllib.request

# Example: public file in S3
bucket_name = "commoncrawl"
object_key = "crawl-data/CC-MAIN-2024-10/warc.paths.gz"
region = "us-east-1"  # some buckets are region-specific

url = f"https://{bucket_name}.s3.{region}.amazonaws.com/{object_key}"
output_file = "warc.paths.gz"

try:
    urllib.request.urlretrieve(url, output_file)
    print(f"✅ Downloaded {object_key} to {output_file}")
except Exception as e:
    print(f"❌ Error downloading file: {e}")


##### Enable Logging of S3

- **Enable Logging with Boto3**

    1. **Import Boto3 and Set Up Client**

    ```python
    import boto3

    # Optionally specify a region
    s3_client = boto3.client('s3', region_name='us-east-1')
    ```


    1. **Define Variables**

    ```python
    source_bucket = 'my-source-bucket'  # The bucket you want to enable logging for
    target_bucket = 'my-log-bucket'     # The bucket where logs will be stored
    log_prefix = 'logs/'                # Optional prefix (folder) for log files
    ```


    1. **Grant Permission to Target Bucket**

    Make sure `my-log-bucket` has a **bucket policy** like this (manually add in AWS Console or automate via Boto3):

    ```json
    {
      "Version": "2012-10-17",
      "Statement": [
        {
          "Sid": "S3ServerAccessLogsPolicy",
          "Effect": "Allow",
          "Principal": {
            "Service": "logging.s3.amazonaws.com"
          },
          "Action": "s3:PutObject",
          "Resource": "arn:aws:s3:::my-log-bucket/logs/*",
          "Condition": {
            "StringEquals": {
              "aws:SourceAccount": "123456789012"
            }
          }
        }
      ]
    }
    ```

    Replace:

        * `my-log-bucket` with your logging bucket
        * `123456789012` with your AWS account ID


    2. **Enable Logging via Boto3**

    ```python
    response = s3_client.put_bucket_logging(
        Bucket=source_bucket,
        BucketLoggingStatus={
            'LoggingEnabled': {
                'TargetBucket': target_bucket,
                'TargetPrefix': log_prefix
            }
        }
    )

    print("Logging enabled:", response)
    ```


- Optional: **Verify Logging is Enabled**

    ```python
    status = s3_client.get_bucket_logging(Bucket=source_bucket)
    print("Current Logging Status:", status)
    ```


- **Notes**

    | Topic              | Detail                                                                                                                   |
    | ------------------ | ------------------------------------------------------------------------------------------------------------------------ |
    | Permissions        | The IAM principal running the script must have `s3:PutBucketLogging`, `s3:GetBucketLogging`, and access to both buckets. |
    | Delay              | Logs typically appear in the logging bucket after a short delay.                                                         |
    | Cost               | Server access logging is free, but you pay for storage of log files.                                                     |
    | Format             | Logs are stored in W3C-like plain text format.                                                                           |
    | Bucket Region Rule | Both source and target buckets must be in the **same region**.                                                           |
